# Домашнее задание по теме "Рекурентные сети 3"

## Задание

1. Возьмите англо-русскую пару фраз ([www.manythings.org....org/anki/](https://www.manythings.org/anki/))
1. Обучите на них seq2seq по аналогии с занятием. Оцените полученное качество
1. Попробуйте добавить +1 рекуррентный слой в encoder и decoder
1. Попробуйте заменить GRU ячейки на lstm-ячейки
1. Оцените качество во всех случаях

In [1]:
from io import open
import unicodedata
import string
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Prepare Data

In [ ]:
!mv ~/Downloads/rus-eng.zip datas/

In [5]:
!unzip ../datas/rus-eng.zip -d ../datas/

Archive:  ../datas/rus-eng.zip
  inflating: ../datas/rus.txt        
  inflating: ../datas/_about.txt     


In [6]:
!rm ../datas/rus-eng.zip

In [7]:
!tail ../datas/rus.txt

I've heard that you should never date anyone who is less than half your age plus seven. Tom is now 30 years old and Mary is 17. How many years will Tom need to wait until he can start dating Mary?	Я слышал, что никогда не следует встречаться с кем-то вдвое младше вас плюс семь лет. Тому 30 лет, a Мэри 17. Сколько лет Тому нужно ждать до тех пор, пока он сможет начать встречаться с Мэри?	CC-BY 2.0 (France) Attribution: tatoeba.org #10068197 (CK) & #10644473 (notenoughsun)
I do have one final ask of you as your president, the same thing I asked when you took a chance on me eight years ago. I'm asking you to believe, not in my ability to bring about change but in yours.	У меня же, как у вашего президента, есть к вам последняя просьба. Та же самая, что и восемь лет назад, когда вы оказали мне своё доверие. Я прошу вас верить, но не в мои способности добиться перемен, а в ваши.	CC-BY 2.0 (France) Attribution: tatoeba.org #5762723 (BHO) & #6390123 (odexed)
In today's world, we have to equip 

In [17]:
SOS_token = 0
EOS_token = 1


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [18]:
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    # Добавляем пробел перед знаками препинания
    s = re.sub(r"([.!?])", r" \1", s)
    # Убираем всё, кроме латиницы, кириллицы и знаков препинания
    s = re.sub(r"[^a-zA-Zа-яА-Я.!?]+", r" ", s)
    return s.strip()

In [19]:
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Если файл называется rus.txt, используем просто имя. 
    # Если хочешь динамически: '../datas/%s-%s.txt' % (lang1, lang2)
    lines = open('../datas/rus.txt', encoding='utf-8').read().strip().split('\n')

    # ИСПРАВЛЕНО: Берем только первые две колонки [:2], игнорируя метаданные
    pairs = [[normalizeString(s) for s in l.split('\t')[:2]] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [20]:
MAX_LENGTH = 10

# Эти префиксы важны для фильтрации датасета, чтобы модель училась на простых фразах
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    # p[0] - входной язык, p[1] - выходной (английский, если reverse=True)
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [21]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

# ВАЖНО: Если файл rus.txt, то lang2 должен быть 'rus'
input_lang, output_lang, pairs = prepareData('eng', 'rus', True)
print(random.choice(pairs))

Reading lines...
Read 536124 sentence pairs
Trimmed to 30628 sentence pairs
Counting words...
Counted words:
rus 10313
eng 4289
['ты к этому не готова .', 'you re not ready for this .']


### 2. Обучение модели с одним рекуррентным слоем

### The Encoder

In [30]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

### The Decoder

In [22]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [23]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [24]:
teacher_forcing_ratio = 0.5


def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(
            input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [25]:
import time
import math


def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [26]:
def trainIters(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [27]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np


def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [28]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words

In [29]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words = evaluate(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [33]:
import torch
import gc

def cleanup_gpu():
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory cleared. Current allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [31]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder1 = DecoderRNN(hidden_size, output_lang.n_words).to(device)

trainIters(encoder1, decoder1, 75000, print_every=5000)

0m 25s (- 5m 55s) (5000 6%) 3.1014
0m 48s (- 5m 16s) (10000 13%) 2.6123
1m 11s (- 4m 47s) (15000 20%) 2.3518
1m 35s (- 4m 22s) (20000 26%) 2.1384
1m 58s (- 3m 56s) (25000 33%) 2.0023
2m 21s (- 3m 32s) (30000 40%) 1.8876
2m 44s (- 3m 8s) (35000 46%) 1.7818
3m 8s (- 2m 44s) (40000 53%) 1.6439
3m 31s (- 2m 21s) (45000 60%) 1.6084
3m 55s (- 1m 57s) (50000 66%) 1.5076
4m 18s (- 1m 34s) (55000 73%) 1.4225
4m 41s (- 1m 10s) (60000 80%) 1.3765
5m 5s (- 0m 47s) (65000 86%) 1.3377
5m 29s (- 0m 23s) (70000 93%) 1.2684
5m 52s (- 0m 0s) (75000 100%) 1.2130


In [32]:
evaluateRandomly(encoder1, decoder1)

> мы завтра встречаемся с томом в центре .
= i m going to meet tom downtown tomorrow .
< i m going to see tomorrow tomorrow . <EOS>

> я слушаю преподавателя .
= i m listening to the teacher .
< i m listening to the . <EOS>

> я все еще голодныи .
= i m still hungry .
< i m still hungry . <EOS>

> я рад что не могу поити .
= i m glad i can t go .
< i m glad we can t go . <EOS>

> он беден но счастлив .
= he is poor but happy .
< he is poor but happy . <EOS>

> я уверен что том не станет возражать .
= i m sure tom won t mind .
< i m sure tom can t swim . <EOS>

> нам за это не платят .
= we re not getting paid for this .
< we re not paid paid for this . <EOS>

> они дома .
= they re home .
< they re home . <EOS>

> я рада что я здесь .
= i m glad to be here .
< i m glad to be here . <EOS>

> ты идеальныи .
= you re perfect .
< you re perfect . <EOS>



In [34]:
cleanup_gpu()

GPU memory cleared. Current allocated: 64.12 MB


## 3. Добавление по одному рекуррентному слою в енкодет и декодер

### The Encoder

In [35]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers=2): # Добавили n_layers
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers # Сохраняем количество слоев

        self.embedding = nn.Embedding(input_size, hidden_size)
        # Указываем num_layers в GRU
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers=n_layers)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        # ВАЖНО: первый размер тензора теперь n_layers
        # Размерность: (n_layers, batch_size, hidden_size)
        return torch.zeros(self.n_layers, 1, self.hidden_size, device=device)

### The Decoder

In [37]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, n_layers=2):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers=n_layers) # Здесь тоже n_layers
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        # Размерность: (n_layers, batch_size, hidden_size)
        return torch.zeros(self.n_layers, 1, self.hidden_size, device=device)

In [38]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder1 = DecoderRNN(hidden_size, output_lang.n_words).to(device)

trainIters(encoder1, decoder1, 75000, print_every=5000)

0m 32s (- 7m 28s) (5000 6%) 3.1220
1m 3s (- 6m 53s) (10000 13%) 2.6524
1m 34s (- 6m 19s) (15000 20%) 2.4260
2m 6s (- 5m 46s) (20000 26%) 2.2198
2m 37s (- 5m 14s) (25000 33%) 2.0564
3m 9s (- 4m 43s) (30000 40%) 1.9016
3m 41s (- 4m 12s) (35000 46%) 1.7759
4m 12s (- 3m 40s) (40000 53%) 1.6781
4m 43s (- 3m 9s) (45000 60%) 1.5967
5m 14s (- 2m 37s) (50000 66%) 1.5105
5m 46s (- 2m 5s) (55000 73%) 1.4498
6m 18s (- 1m 34s) (60000 80%) 1.3369
6m 50s (- 1m 3s) (65000 86%) 1.3311
7m 21s (- 0m 31s) (70000 93%) 1.2630
7m 53s (- 0m 0s) (75000 100%) 1.1887


In [39]:
evaluateRandomly(encoder1, decoder1)

> я боюсь что мы не сможем остаться тут .
= i m afraid we can t stay here .
< i m afraid we can can stay here . <EOS>

> я начинаю понимать .
= i am beginning to understand .
< i m starting to lose . <EOS>

> вода !
= you re it !
< you re so dirty . <EOS>

> я не беден .
= i m not poor .
< i m not poor . <EOS>

> вы сильны .
= you re powerful .
< you re clever . <EOS>

> он смелыи и веселыи мальчик .
= he is a brave and cheerful boy .
< he is a and and boy . <EOS>

> он отличается от своего старшего брата .
= he is different from his older brother .
< he is only different from his brother . <EOS>

> вы совершенно бесполезны .
= you re completely useless .
< you re completely normal . <EOS>

> мы тут .
= we re here .
< we re here . <EOS>

> можешь брать мою машину .
= you re welcome to borrow my car .
< you re welcome to go our car . <EOS>



In [40]:
cleanup_gpu()

GPU memory cleared. Current allocated: 72.96 MB


## 4. LSTM

### The Encoder

In [42]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers=2):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_size, hidden_size)
        # Заменяем GRU на LSTM
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_layers)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        # hidden теперь это кортеж (h, c)
        output, hidden = self.lstm(embedded, hidden)
        return output, hidden

    def initHidden(self):
        # Для LSTM нужно инициализировать ДВА тензора: h и c
        h_0 = torch.zeros(self.n_layers, 1, self.hidden_size, device=device)
        c_0 = torch.zeros(self.n_layers, 1, self.hidden_size, device=device)
        return (h_0, c_0) # Возвращаем кортеж

### The Decoder

In [43]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, n_layers=2):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_size, hidden_size)
        # Заменяем GRU на LSTM
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_layers)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        # Передаем и получаем кортеж (h, c)
        output, hidden = self.lstm(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        h_0 = torch.zeros(self.n_layers, 1, self.hidden_size, device=device)
        c_0 = torch.zeros(self.n_layers, 1, self.hidden_size, device=device)
        return (h_0, c_0)

In [44]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder1 = DecoderRNN(hidden_size, output_lang.n_words).to(device)

trainIters(encoder1, decoder1, 75000, print_every=5000)

0m 35s (- 8m 19s) (5000 6%) 3.3145
1m 10s (- 7m 38s) (10000 13%) 2.8711
1m 45s (- 7m 3s) (15000 20%) 2.6720
2m 21s (- 6m 27s) (20000 26%) 2.4983
2m 56s (- 5m 53s) (25000 33%) 2.3918
3m 32s (- 5m 18s) (30000 40%) 2.2452
4m 7s (- 4m 43s) (35000 46%) 2.1515
4m 43s (- 4m 7s) (40000 53%) 2.0566
5m 18s (- 3m 32s) (45000 60%) 1.9450
5m 53s (- 2m 56s) (50000 66%) 1.8659
6m 28s (- 2m 21s) (55000 73%) 1.7973
7m 4s (- 1m 46s) (60000 80%) 1.7140
7m 39s (- 1m 10s) (65000 86%) 1.6225
8m 15s (- 0m 35s) (70000 93%) 1.5691
8m 50s (- 0m 0s) (75000 100%) 1.4837


In [45]:
evaluateRandomly(encoder1, decoder1)

> я рад что вы это сказали .
= i m glad you said that .
< i m glad you brought that . <EOS>

> я шокирована .
= i m shocked .
< i m a . . <EOS>

> ты опоздал .
= you are late .
< you re late late . <EOS>

> они сеичас уидут .
= they re about to leave .
< they re leaving now . <EOS>

> я отчасти виноват в случившемся .
= i m partly responsible for what happened .
< i m responsible for for alone . <EOS>

> мы с ним примерно ровесники .
= he is about my age .
< he s about the same age as <EOS>

> я в ожидании очень важного звонка .
= i m waiting for a very important call .
< i m very to to a tomorrow . <EOS>

> ты терпелив .
= you re patient .
< you re vain . <EOS>

> я не умею разговаривать с людьми .
= i m not good at talking to people .
< i m not good good at . . <EOS>

> вы слишком скромная .
= you re too humble .
< you re too too . <EOS>



In [46]:
cleanup_gpu()

GPU memory cleared. Current allocated: 78.61 MB


## Оценка качества моделей:

1. Усложнение архитектуры замедлило процесс.
    - GRU (1 слой) справилась за 6 минут, в то время как LSTM (2 слоя) потребовалось почти 9 минут.
    - При этом на данном этапе GRU оказалась эффективнее: она быстрее достигла низкого уровня потерь (loss 1.21 против 1.48 у LSTM за те же 75к итераций).
1. Для этой конкретной задачи (короткие фразы с фиксированными префиксами) GRU с 2 слоями показала лучший результат (самый низкий loss — 1.18).
1. LSTM, вероятно, требует больше времени "на раскачку". Заметил, что loss у LSTM в конце всё еще бодро шел вниз? Это значит, что 75 000 итераций ей просто не хватило.
1. Качество перевода: LSTM, несмотря на более высокий loss, иногда выдает более человечные варианты. Например, вместо точного «about to leave» она перевела «leaving now». Смысл сохранен, хотя слова другие. Это говорит о том, что модель начала схватывать контекст, но ей не хватает точности.